In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler  # 数据标准化提升SVM效果
import joblib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3D绘图依赖
from matplotlib.colors import ListedColormap
from matplotlib.table import Table  # 表格绘制依赖

# ===================== 第一步：数据读取与预处理（提升准确率核心步骤） =====================
# 读取训练用的两个文件（确保文件路径正确）
df1 = pd.read_csv('country_two_years_participant_times.csv')
df2 = pd.read_csv('only_zero_gold_countries_with_participants.csv')

# 提取特征和标签
X1 = df1.iloc[:, [3, 4, 7]].values  # 原文件4、5、8列（第一维=铜牌，第二维=银牌，第三维=其他）
X2 = df2.iloc[:, [5, 6, 8]].values  # 原文件6、7、9列（第一维=铜牌，第二维=银牌，第三维=其他）
y1 = np.ones(X1.shape[0])  # S类（1）
y2 = np.zeros(X2.shape[0]) # F类（0）

# 合并训练数据
X = np.concatenate((X1, X2), axis=0)
y = np.concatenate((y1, y2), axis=0)

# 🌟 关键优化1：数据标准化（SVM对特征尺度敏感，标准化能大幅提升准确率）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # 标准化为均值0、方差1

# 划分训练集和测试集（标准化后的数据）
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y  # stratify：分层抽样，保证类别分布一致
)

# ===================== 第二步：SVM超参数调优（提升准确率核心步骤） =====================
# 🌟 关键优化2：网格搜索最优超参数（代替固定C和gamma）
param_grid = {
    'C': [0.1, 1, 10, 100],  # 惩罚系数：越大对误分类惩罚越重
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  # 核函数系数：影响RBF核的拟合程度
    'kernel': ['rbf']  # 保持RBF核，适合非线性数据
}

# 网格搜索+交叉验证（cv=5：5折交叉验证）
grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),  # probability=True用于后续可视化
    param_grid, 
    cv=5, 
    scoring='accuracy',  # 以准确率为评价指标
    n_jobs=-1  # 利用所有CPU核心加速
)
grid_search.fit(X_train, y_train)

# 输出最优参数和训练集最优准确率
print("🌟 最优超参数：", grid_search.best_params_)
print("🌟 交叉验证最优准确率：{:.2f}%".format(grid_search.best_score_ * 100))

# 使用最优参数构建最终模型
svm_model = grid_search.best_estimator_

# 在测试集评估模型效果
y_pred = svm_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print("🌟 测试集最终准确率：{:.2f}%".format(test_accuracy * 100))
print("\n分类报告：")
print(classification_report(y_test, y_pred))

# 保存模型和标准化器（后续预测需要用相同的scaler）
joblib.dump(svm_model, 'optimized_svm_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("\n优化后的模型和标准化器已保存！")


In [ ]:
# ===================== 第三步：训练数据三维散点图可视化 =====================
# 设置可视化样式
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
cmap = ListedColormap(['#FF0000', '#00FF00'])  # 红=F(0)，绿=S(1)

# 创建3D图
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 绘制训练数据三维散点图（用标准化前的数据更直观）
scatter = ax.scatter(
    X[:, 0], X[:, 1], X[:, 2],  # 三个特征维度
    c=y,  # 按类别着色
    cmap=cmap,
    edgecolors='k',  # 黑色边框，增强区分度
    s=10,  # 点的大小
    alpha=0.8  # 半透明，避免重叠遮挡
)

# 设置坐标轴标签
ax.set_xlabel('特征1（铜牌）', fontsize=12)
ax.set_ylabel('特征2（银牌）', fontsize=12)
ax.set_zlabel('特征3（第8/9列）', fontsize=12)
ax.set_title('SVM训练数据三维散点图（F=红色，S=绿色）', fontsize=14)

# 添加图例
legend1 = ax.legend(*scatter.legend_elements(), title="类别")
ax.add_artist(legend1)
plt.tight_layout()
plt.savefig('train_data_3d_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 设置全局字体为Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.sans-serif'] = ['Times New Roman']

# ===================== 1. 数据统计与交叉表构建（核心：互换行列） =====================
bronze_silver_df = pd.DataFrame({
    '铜牌数量': X[:, 0].astype(int),
    '银牌数量': X[:, 1].astype(int),
    '类别': np.where(y == 1, 'S', 'F')
})

# 统计每个组合的S数量和总数量
total_count = bronze_silver_df.groupby(['铜牌数量', '银牌数量']).size().rename('总数量')
s_count = bronze_silver_df[bronze_silver_df['类别'] == 'S'].groupby(['铜牌数量', '银牌数量']).size()
s_count = s_count.reindex(total_count.index, fill_value=0).rename('S数量')

# 计算占比并构建交叉表（核心：互换行列→横坐标=铜牌，纵坐标=银牌）
ratio_df = pd.concat([s_count, total_count], axis=1)
ratio_df['占比(%)'] = (ratio_df['S数量'] / ratio_df['总数量'] * 100).round(2)

# 生成完整连续的坐标（横坐标=铜牌0-7，纵坐标=银牌0-6）
bronze_vals = np.arange(0, 8)  # 横坐标：铜牌 0-7
silver_vals = np.arange(0, 7)  # 纵坐标：银牌 0-6

# 核心互换：原本行=铜牌、列=银牌 → 现在行=银牌、列=铜牌（实现坐标互换）
cross_ratio = ratio_df['占比(%)'].unstack(fill_value=0)  # 原始：列=银牌，行=铜牌
cross_ratio = cross_ratio.T  # 转置！实现行列互换 → 列=铜牌，行=银牌
cross_ratio = cross_ratio.reindex(index=silver_vals, columns=bronze_vals, fill_value=0)

# ===================== 2. 绘制表格（坐标互换+超大字体） =====================
fig, ax = plt.subplots(figsize=(18, 8))  # 适配横坐标更长的布局
ax.set_axis_off()

# 表格布局参数
cell_w = 1.0    # 单元格宽度（横坐标更长，保持宽度）
cell_h = 0.8    # 单元格高度
x0 = 1
y0 = 1
border = 1.5
inner_border = 1.0

# 蓝色渐变配色
def get_color(ratio):
    if ratio == 0:
        return (1, 1, 1)  # 0%为白色
    r = 1 - (ratio / 100 * 0.6)
    g = 1 - (ratio / 100 * 0.4)
    b = 1
    return (r, g, b)

# 绘制单元格（数字放大到16号，加粗）
# 外层循环：纵坐标=银牌（行），内层循环：横坐标=铜牌（列）
for i, silver in enumerate(cross_ratio.index):  # i=纵坐标（银牌）
    for j, bronze in enumerate(cross_ratio.columns):  # j=横坐标（铜牌）
        x = x0 + j * cell_w
        y = y0 + i * cell_h
        ratio_val = cross_ratio.loc[silver, bronze]
        color = get_color(ratio_val)

        # 绘制单元格背景+边框
        cell = Rectangle(
            (x, y), cell_w, cell_h,
            facecolor=color,
            edgecolor='black',
            linewidth=inner_border,
            zorder=1
        )
        ax.add_patch(cell)

        # 百分比数字（16号超大字体，加粗）
        if ratio_val > 0:
            ax.text(
                x + cell_w/2, y + cell_h/2, f"{ratio_val:.2f}%",
                ha='center', va='center', fontsize=24, zorder=2
            )

# 绘制表格外框
table_w = len(cross_ratio.columns) * cell_w  # 宽度=铜牌数量
table_h = len(cross_ratio.index) * cell_h    # 高度=银牌数量
outer_box = Rectangle(
    (x0, y0), table_w, table_h,
    facecolor='none',
    edgecolor='black',
    linewidth=border,
    zorder=3
)
ax.add_patch(outer_box)

# 添加坐标轴刻度（坐标互换+极近表格+14号字体）
# X轴（横坐标）：铜牌数量 0-7（极近表格，偏移0.15）
for j, bronze in enumerate(cross_ratio.columns):
    ax.text(
        x0 + j*cell_w + cell_w/2, y0 - 0.2,
        str(bronze), ha='center', va='center', fontsize=24, 
    )
# Y轴（纵坐标）：银牌数量 0-6（极近表格，偏移0.15）
for i, silver in enumerate(cross_ratio.index):
    ax.text(
        x0 - 0.1, y0 + i*cell_h + cell_h/2,
        str(silver), ha='center', va='center', fontsize=24, 
    )

# 调整图表范围（紧凑布局）
ax.set_xlim(x0 - 0.3, x0 + table_w + 0.3)
ax.set_ylim(y0 - 0.3, y0 + table_h + 0.3)
ax.set_title('Bronze-Silver Combination S Ratio Table', fontsize=30, pad=20 )

# 保存高清图片
plt.tight_layout()
plt.savefig('final_table_correct_axis.png', dpi=300, bbox_inches='tight')
plt.show()

# 打印文本版表格（验证坐标互换）
print("========== 坐标互换后：铜牌(横坐标)-银牌(纵坐标) S类占比（%）文本表格 ==========")
print(cross_ratio)